In [4]:
import os
import numpy as np
import cv2
from tqdm import tqdm # İlerleme çubuğu için güzel bir kütüphane

# --- AYARLAR ---
# YOLO formatındaki etiket dosyalarının bulunduğu GİRDİ klasörü
YOLO_LABELS_DIR = r"C:\Users\524ha\.cache\kagglehub\datasets\sallerroig\court-seg\versions\1\valid\labels"

# Oluşturulacak maskelerin kaydedileceği ÇIKTI klasörü
OUTPUT_MASKS_DIR = 'masks'

# Görüntülerin ve oluşturulacak maskelerin boyutları
IMAGE_WIDTH = 640
IMAGE_HEIGHT = 640
# --- AYARLAR SONU ---


# --- HİYERARŞİ VE SINIF TANIMLAMALARI (Öncekiyle aynı) ---
mask_class_ids = {
    "ARKA_PLAN": 0,
    "UC_SAYI_BOLGESI": 1,
    "IKI_SAYI_BOLGESI": 2,
    "UC_SANIYE_ALANI": 3
}

yolo_id_to_mask_id = {
    0: mask_class_ids['UC_SAYI_BOLGESI'],
    2: mask_class_ids['IKI_SAYI_BOLGESI'],
    1: mask_class_ids['UC_SANIYE_ALANI']
}

drawing_order_yolo_ids = [0, 2, 1]
# --- HİYERARŞİ SONU ---


def create_mask_from_yolo_file(txt_path, width, height):
    """
    Tek bir YOLO .txt dosyasından hiyerarşik segmentasyon maskesi oluşturur.
    (Bu fonksiyon önceki script ile birebir aynıdır)
    """
    polygons_by_yolo_id = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3: continue
                
                class_id = int(parts[0])
                coords = np.array([float(p) for p in parts[1:]])
                coords[0::2] *= width
                coords[1::2] *= height
                polygon = coords.reshape(-1, 2).astype(np.int32)
                polygons_by_yolo_id[class_id] = polygon
    except FileNotFoundError:
        print(f"Hata: {txt_path} bulunamadı.")
        return None
    
    mask = np.full((height, width), mask_class_ids["ARKA_PLAN"], dtype=np.uint8)

    for yolo_id in drawing_order_yolo_ids:
        if yolo_id in polygons_by_yolo_id:
            polygon_to_draw = polygons_by_yolo_id[yolo_id]
            mask_id_to_paint = yolo_id_to_mask_id[yolo_id]
            cv2.fillPoly(mask, [polygon_to_draw], mask_id_to_paint)
            
    return mask

# --- ANA ÇALIŞTIRMA KISMI (TOPLU İŞLEME) ---
if __name__ == "__main__":
    # Girdi klasörünün var olup olmadığını kontrol et
    if not os.path.isdir(YOLO_LABELS_DIR):
        print(f"HATA: Girdi klasörü bulunamadı: '{YOLO_LABELS_DIR}'")
        exit()

    # Çıktı klasörü yoksa oluştur
    os.makedirs(OUTPUT_MASKS_DIR, exist_ok=True)

    # Girdi klasöründeki tüm .txt dosyalarını bul
    label_files = [f for f in os.listdir(YOLO_LABELS_DIR) if f.endswith('.txt')]
    
    if not label_files:
        print(f"UYARI: '{YOLO_LABELS_DIR}' klasöründe işlenecek .txt dosyası bulunamadı.")
        exit()

    print(f"Toplam {len(label_files)} adet etiket dosyası bulundu. Maskeler oluşturuluyor...")

    # Her bir etiket dosyası için döngü başlat
    for filename in tqdm(label_files, desc="Maskeler Oluşturuluyor"):
        # Dosya yollarını oluştur
        input_path = os.path.join(YOLO_LABELS_DIR, filename)
        
        # Çıktı dosya adını belirle (.txt yerine .png)
        output_filename = os.path.splitext(filename)[0] + '.png'
        output_path = os.path.join(OUTPUT_MASKS_DIR, output_filename)
        
        # Etiket dosyasından maskeyi oluştur
        mask = create_mask_from_yolo_file(input_path, IMAGE_WIDTH, IMAGE_HEIGHT)
        
        # Maske başarıyla oluşturulduysa kaydet
        if mask is not None:
            cv2.imwrite(output_path, mask)

    print(f"\nİşlem tamamlandı!")
    print(f"{len(label_files)} adet maske başarıyla '{OUTPUT_MASKS_DIR}' klasörüne kaydedildi.")

Toplam 50 adet etiket dosyası bulundu. Maskeler oluşturuluyor...


Maskeler Oluşturuluyor: 100%|██████████| 50/50 [00:00<00:00, 101.13it/s]


İşlem tamamlandı!
50 adet maske başarıyla 'masks' klasörüne kaydedildi.
